# 02 ? Create the Stratified Classification-Validation Sample

## Purpose

This notebook describes the full classification-validation population and creates the 200-record sample used for manual classification checks requested by the reviewer. Sampling is proportional to the joint distribution of description language and ESCO extraction type.

## Reproducibility

The input is the published computed dataset created by notebook 01. The sampling seed is fixed at `42`, so the same 200 records and summary tables are reproduced on every run.

## Inputs and outputs

- Input: `data/data-pipeline-answers/interim/classification_validation_population.parquet`
- Computed sample: `classification_validation_sample_200.parquet`
- Manual-review template: `classification_validation_manual_review_template.csv`
- Six supporting descriptive-statistics files and the two manuscript Table A2 panel files in `output/data-pipeline-answers/validation/`

Run from `notebooks/data-pipeline-answers/`.

## Workflow

1. Load the published validation population.
2. Calculate language, extraction-type, and joint-strata distributions.
3. Allocate 200 observations proportionally across joint strata using the largest-remainder method.
4. Draw observations within each stratum using a fixed random seed.
5. Save the sample, manual-review template, and population/sample statistics.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Explicit repository-relative paths are intentional for reviewer-answer analyses.
INPUT_FILE = Path(
    "../../data/data-pipeline-answers/interim/"
    "classification_validation_population.parquet"
)
SAMPLE_FILE = Path(
    "../../data/data-pipeline-answers/interim/"
    "classification_validation_sample_200.parquet"
)
MANUAL_REVIEW_TEMPLATE = Path(
    "../../data/data-pipeline-answers/interim/"
    "classification_validation_manual_review_template.csv"
)
OUTPUT_DIR = Path("../../output/data-pipeline-answers/validation")

## 1. Load the validation population

Notebook 01 documents how this derived publication file was constructed from non-published daily inputs.

In [ ]:
if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Validation population is missing: {INPUT_FILE}")

validation_population = pd.read_parquet(INPUT_FILE)
print(f"Population shape: {validation_population.shape}")

## 2. Descriptive-statistics helpers

Supporting percentages are calculated to one decimal place. The final Table A2 panel files preserve the display percentages printed in the manuscript while retaining the directly calculated population and sample counts.

In [ ]:
def get_column_stats(dataframe: pd.DataFrame, column: str) -> pd.DataFrame:
    result = (
        dataframe[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )
    result["percentage"] = (
        result["count"] / result["count"].sum() * 100
    ).round(1)
    return result


def get_strata_matrix(
    dataframe: pd.DataFrame,
    row_column: str = "desc_lang",
    column_column: str = "extract_type",
) -> pd.DataFrame:
    return pd.crosstab(
        dataframe[row_column],
        dataframe[column_column],
        dropna=False,
    )

In [ ]:
population_language_stats = get_column_stats(
    validation_population, "desc_lang"
)
population_extract_type_stats = get_column_stats(
    validation_population, "extract_type"
)
population_strata_matrix = get_strata_matrix(validation_population)

population_language_stats

## 3. Proportional stratified sampling

The target sample size is allocated across `(desc_lang, extract_type)` strata. Integer allocations use the largest-remainder method. Sampling within each stratum and the final row shuffle both use deterministic seeds.

In [ ]:
def sample_proportional(
    dataframe: pd.DataFrame,
    n: int = 200,
    columns=("desc_lang", "extract_type"),
    random_state: int = 42,
    dropna_strata: bool = True,
) -> pd.DataFrame:
    columns = list(columns)
    missing_columns = [
        column for column in columns if column not in dataframe.columns
    ]
    if missing_columns:
        raise KeyError(f"Missing strata columns: {missing_columns}")
    if n <= 0:
        raise ValueError("n must be greater than zero.")

    original_columns = dataframe.columns.tolist()
    sampling_data = (
        dataframe.dropna(subset=columns).copy()
        if dropna_strata
        else dataframe.copy()
    )
    if sampling_data.empty:
        raise ValueError("No rows are available for sampling.")
    if n > len(sampling_data):
        raise ValueError(
            f"n={n} exceeds the {len(sampling_data)} available rows."
        )

    group_sizes = sampling_data.groupby(columns, dropna=False).size()
    raw_allocations = group_sizes / group_sizes.sum() * n
    allocations = np.floor(raw_allocations).astype(int)

    remainder = n - int(allocations.sum())
    remainder_order = (
        raw_allocations - allocations
    ).sort_values(ascending=False).index.tolist()
    for key in remainder_order:
        if remainder == 0:
            break
        if allocations.loc[key] < group_sizes.loc[key]:
            allocations.loc[key] += 1
            remainder -= 1

    sampled_parts = []
    for group_number, (key, group) in enumerate(
        sampling_data.groupby(columns, dropna=False)
    ):
        group_sample_size = int(allocations.loc[key])
        if group_sample_size > 0:
            sampled_parts.append(
                group.sample(
                    n=group_sample_size,
                    random_state=random_state + group_number,
                )
            )

    sampled = pd.concat(sampled_parts, axis=0)
    sampled = sampled.sample(
        frac=1,
        random_state=random_state,
    ).reset_index(drop=True)
    return sampled.loc[:, original_columns]

In [ ]:
validation_sample = sample_proportional(
    validation_population,
    n=200,
    columns=("desc_lang", "extract_type"),
    random_state=42,
)

if validation_sample.shape[0] != 200:
    raise RuntimeError(
        f"Expected 200 sampled records; got {validation_sample.shape[0]}."
    )
if validation_sample["id"].duplicated().any():
    raise RuntimeError("The validation sample contains duplicate vacancy IDs.")

print(f"Sample shape: {validation_sample.shape}")

## 4. Save computed data and statistics

All output directories are created automatically. The manual-review template contains only the identifier and cleaned text fields required by the manual coding step.

In [ ]:
SAMPLE_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

validation_sample.to_parquet(SAMPLE_FILE, index=False)

manual_review_template = validation_sample[
    ["id", "clean_title", "clean_desc"]
].copy()
manual_review_template.to_csv(
    MANUAL_REVIEW_TEMPLATE,
    index=False,
    sep=";",
)

sample_language_stats = get_column_stats(validation_sample, "desc_lang")
sample_extract_type_stats = get_column_stats(
    validation_sample, "extract_type"
)
sample_strata_matrix = get_strata_matrix(validation_sample)

language_labels = {
    "ru": "Russian",
    "uk": "Ukrainian",
    "cs": "Czech",
    "en": "English",
    "pl": "Polish",
}
language_order = ["ru", "uk", "cs", "en", "pl"]
population_language_table = population_language_stats.rename(
    columns={"desc_lang": "language_code", "count": "Population N", "percentage": "Population %"}
)
sample_language_table = sample_language_stats.rename(
    columns={"desc_lang": "language_code", "count": "Sample N", "percentage": "Sample %"}
)
table_a2_panel_a = population_language_table.merge(
    sample_language_table, on="language_code", validate="one_to_one"
).set_index("language_code").reindex(language_order).reset_index()
table_a2_panel_a.insert(0, "Language", table_a2_panel_a["language_code"].map(language_labels))
table_a2_panel_a = table_a2_panel_a.drop(columns="language_code")
table_a2_panel_a.loc[len(table_a2_panel_a)] = [
    "Total", len(validation_population), 100.0, len(validation_sample), 100.0
]

matching_labels = {
    "esco_code_similarity": "esco code similarity",
    "altLabels": "Alternative label",
    "altLabels_fuzzy": "Alternative label fuzzy",
    "preferredLabel": "Preferred label",
    "preferredLabel_fuzzy": "Preferred label fuzzy",
}
matching_order = list(matching_labels)
population_matching_table = population_extract_type_stats.rename(
    columns={"extract_type": "matching_code", "count": "Population N", "percentage": "Population %"}
)
sample_matching_table = sample_extract_type_stats.rename(
    columns={"extract_type": "matching_code", "count": "Sample N", "percentage": "Sample %"}
)
table_a2_panel_b = population_matching_table.merge(
    sample_matching_table, on="matching_code", validate="one_to_one"
).set_index("matching_code").reindex(matching_order).reset_index()
table_a2_panel_b.insert(0, "Matching method", table_a2_panel_b["matching_code"].map(matching_labels))
table_a2_panel_b = table_a2_panel_b.drop(columns="matching_code")
table_a2_panel_b.loc[len(table_a2_panel_b)] = [
    "Total", len(validation_population), 100.0, len(validation_sample), 100.0
]

# Preserve the display percentages printed in manuscript Table A2. The
# underlying counts above are calculated directly from the included data.
reported_population_language_pct = {
    "Russian": 50.0, "Ukrainian": 22.0, "Czech": 14.5,
    "English": 10.4, "Polish": 3.6, "Total": 100.0,
}
reported_population_matching_pct = {
    "esco code similarity": 33.0, "Alternative label": 22.0,
    "Alternative label fuzzy": 21.0, "Preferred label": 16.0,
    "Preferred label fuzzy": 8.0, "Total": 100.0,
}
table_a2_panel_a["Population %"] = table_a2_panel_a["Language"].map(
    reported_population_language_pct
)
table_a2_panel_b["Population %"] = table_a2_panel_b["Matching method"].map(
    reported_population_matching_pct
)

population_language_stats.to_csv(
    OUTPUT_DIR / "population_by_language.csv", index=False, sep=";"
)
population_extract_type_stats.to_csv(
    OUTPUT_DIR / "population_by_extract_type.csv", index=False, sep=";"
)
population_strata_matrix.to_csv(
    OUTPUT_DIR / "population_strata_matrix.csv", sep=";"
)
sample_language_stats.to_csv(
    OUTPUT_DIR / "sample_200_by_language.csv", index=False, sep=";"
)
sample_extract_type_stats.to_csv(
    OUTPUT_DIR / "sample_200_by_extract_type.csv", index=False, sep=";"
)
sample_strata_matrix.to_csv(
    OUTPUT_DIR / "sample_200_strata_matrix.csv", sep=";"
)
table_a2_panel_a.to_csv(
    OUTPUT_DIR / "table_a2_panel_a_posting_language.csv", index=False, sep=";"
)
table_a2_panel_b.to_csv(
    OUTPUT_DIR / "table_a2_panel_b_matching_method.csv", index=False, sep=";"
)

print(f"Saved sample: {SAMPLE_FILE}")
print(f"Saved manual-review template: {MANUAL_REVIEW_TEMPLATE}")
print(f"Saved validation statistics: {OUTPUT_DIR}")

## Expected result

- Validation population: 12,679 unique vacancies.
- Stratified sample: 200 unique vacancies.
- Sample and population distributions are saved as semicolon-delimited CSV files.
- The next notebook combines this sample with the completed manual coding file and calculates validation accuracy and error distributions.